# PolicyRec v1.1.5 — 룰표 v2 적용 + dedupe + 임베딩 준비

## 이 노트북의 역할

`raw_v1_1_5.csv` (600건) → 임베딩 단계로 들어갈 수 있는 깨끗한 main_csv 생성.

### 처리 순서

1. **raw 로드 + 정규화** (제목/기관/기간/URL)
2. **카테고리 룰표 v2 적용** → `s_category` 부여
3. **스코프 룰표 v2 적용** → `_scope`, `_scope_reason` 부여
4. **dedupe 처리**
   - 강한 신호 (URL exact) → 자동 other (`duplicate_url`)
   - 완전 동일 (제목+기관+기간) → 자동 other (`duplicate_exact`)
   - 자매 공고 (애매한 케이스) → review 큐
5. **main_csv 출력** (임베딩용 깨끗한 데이터)
6. **review 큐 csv 출력** (사람 검토용)

## v1 대비 v2 변경점

| 항목 | v1 | v2 |
| :--- | :--- | :--- |
| civic 처리 | 키워드 8개 | 카테고리 룰 (`참여/기반` 자동) + 보강 키워드 5개 |
| 위험 키워드 | 경진대회/박람회/페스티벌/서포터즈 포함 | **모두 제거** (창업지원사업이라 main 유지해야 함) |
| dedupe | review 큐만 분리 | 자동 처리 + review 큐 분리 |
| 중복 카테고리 | 처리 안 됨 | youth `참여･기반,참여･기반` 등 처리

## 0. 셋업

In [1]:
from pathlib import Path
import re
import html
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(v): print(v)

# ============================================================
# 경로 설정 (팀원 v1.1.4 폴더 구조 그대로)
# ============================================================
PROJECT_ROOT = Path.cwd()
VERSION = "v1_1_5"


CSV_ROOT      = PROJECT_ROOT / "data" / "csv"
CSV_RAW_DIR   = CSV_ROOT / "raw"
CSV_MAIN_DIR  = CSV_ROOT / "main"
CSV_RULE_DIR  = CSV_ROOT / "rule"
CSV_REVIEW_DIR = CSV_ROOT / "review"
for d in [CSV_MAIN_DIR, CSV_RULE_DIR, CSV_REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 입력
RAW_CSV       = CSV_RAW_DIR  / f"raw_{VERSION}.csv"

# 룰표 v2
CAT_RULE_CSV   = CSV_RULE_DIR / f"cat_rule_{VERSION}.csv"
SCOPE_RULE_CSV = CSV_RULE_DIR / f"scope_rule_{VERSION}.csv"

# 출력
MAIN_CSV       = CSV_MAIN_DIR  / f"main_{VERSION}.csv"
REVIEW_CSV     = CSV_REVIEW_DIR / f"review_queue_{VERSION}.csv"

print("[입력]")
print(f"  raw           : {RAW_CSV}")
print(f"  cat_rule v2   : {CAT_RULE_CSV}")
print(f"  scope_rule v2 : {SCOPE_RULE_CSV}")
print()
print("[출력]")
print(f"  main (임베딩용) : {MAIN_CSV}")
print(f"  review queue   : {REVIEW_CSV}")

[입력]
  raw           : c:\Users\CY2\github\policyrec\PolicyRec\data\csv\raw\raw_v1_1_5.csv
  cat_rule v2   : c:\Users\CY2\github\policyrec\PolicyRec\data\csv\rule\cat_rule_v1_1_5.csv
  scope_rule v2 : c:\Users\CY2\github\policyrec\PolicyRec\data\csv\rule\scope_rule_v1_1_5.csv

[출력]
  main (임베딩용) : c:\Users\CY2\github\policyrec\PolicyRec\data\csv\main\main_v1_1_5.csv
  review queue   : c:\Users\CY2\github\policyrec\PolicyRec\data\csv\review\review_queue_v1_1_5.csv


## 1. raw csv 로드 + 기본 정리

In [2]:
df = pd.read_csv(RAW_CSV, dtype=str).fillna("")
print(f"raw shape: {df.shape}")

# HTML entity 풀기 (제목/요약/기관)
for col in ["title", "summary", "supervising_agency", "operating_agency", "category", "subcategory"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda v: html.unescape(str(v)).strip())

# provider: supervising_agency 우선, 없으면 operating_agency
df["provider"] = df["supervising_agency"].where(
    df["supervising_agency"].astype(str).str.strip() != "",
    df["operating_agency"]
)

print()
print("=== source 분포 ===")
display(df["source"].value_counts().rename_axis("source").reset_index(name="count"))
print()
print("=== category 분포 (raw) ===")
display(df.groupby(["source", "category"]).size().reset_index(name="count"))

raw shape: (600, 22)

=== source 분포 ===


,source,count
0,biz,200
1,kst,200
2,youth,200



=== category 분포 (raw) ===


,source,category,count
0,biz,경영,64
1,biz,금융,8
2,biz,기술,39
3,biz,기타,2
4,biz,내수,11
5,biz,수출,41
6,biz,인력,19
7,biz,창업,16
8,kst,글로벌,10
9,kst,기술개발(R&D),4


## 2. 카테고리 룰표 v2 적용

raw `category` → `s_category` 매핑.
중복 카테고리 표기(`참여･기반,참여･기반`)는 룰표에 명시된 매핑 사용.

In [3]:
cat_rule = pd.read_csv(CAT_RULE_CSV, comment="#", dtype=str).fillna("")
print(f"카테고리 룰표 shape: {cat_rule.shape}")
display(cat_rule)

# 룰표를 dict로 변환: (source, raw_category) → s_category
rule_map = {
    (row["source"], row["raw_category"]): row["s_category"]
    for _, row in cat_rule.iterrows()
}

# 적용
def apply_cat_rule(row):
    key = (row["source"], row["category"])
    return rule_map.get(key, "기타")  # 룰표에 없으면 기타로 fallback

df["s_category"] = df.apply(apply_cat_rule, axis=1)

# 미매핑 케이스 중 중복 카테고리("A,A" 형태) 처리
# ex) "참여･기반,참여･기반" → "참여･기반"으로 정규화 후 재시도
unmapped_mask = df["s_category"] == "기타"
for idx in df[unmapped_mask].index:
    raw_cat = df.loc[idx, "category"]
    if "," in raw_cat:
        # 중복 제거: "A,A" → "A"
        parts = [p.strip() for p in raw_cat.split(",")]
        deduped = parts[0] if len(set(parts)) == 1 else raw_cat
        key = (df.loc[idx, "source"], deduped)
        if key in rule_map:
            df.loc[idx, "s_category"] = rule_map[key]

# 최종 미매핑 확인
unmapped = df[df["s_category"] == "기타"]
unmapped_sources = unmapped.groupby(["source", "category"]).size().reset_index(name="count")
unmapped_sources = unmapped_sources[unmapped_sources["count"] > 0]

print()
print(f"=== 룰표 적용 결과: 매핑된 {(df['s_category']!='기타').sum()}건 / 미매핑 {len(unmapped)}건 ===")
if len(unmapped):
    print("\n룰표에 없는 (source, category) 조합:")
    display(unmapped_sources)

print()
print("=== s_category × source 교차표 ===")
display(pd.crosstab(df["s_category"], df["source"], margins=True, margins_name="합계"))

카테고리 룰표 shape: (23, 4)


,source,raw_category,s_category,note
0,biz,창업,창업,창업 지원
1,biz,경영,경영,carry-all (subcategory: 사업화37/컨설팅20/시설지원6/교육1)
2,biz,기술,기술,R&D 기술개발
3,biz,인력,인력/일자리,인력 채용/일자리 지원
4,biz,수출,판로/수출,해외 진출
5,biz,내수,판로/수출,국내 시장
6,biz,금융,자금,융자/보증
7,biz,기타,기타,분류 불명확
8,kst,사업화,창업,창업 사업화
9,kst,창업교육,창업,창업 교육



=== 룰표 적용 결과: 매핑된 598건 / 미매핑 2건 ===

룰표에 없는 (source, category) 조합:


,source,category,count
0,biz,기타,2



=== s_category × source 교차표 ===


source,biz,kst,youth,합계
s_category,,,,
경영,64,0,0,64
교육/멘토링,0,44,18,62
기술,39,4,0,43
기타,2,0,0,2
복지/문화,0,0,60,60
시설/공간,0,31,0,31
인력/일자리,19,2,65,86
자금,8,1,0,9
주거,0,0,23,23


## 3. norm_* 컬럼 생성 (dedupe용)

비교 정확도를 위해 공백/특수문자 제거한 norm 버전 생성.

In [4]:
_RX_NON = re.compile(r"[\s\W_]+", flags=re.UNICODE)

def normalize_text(s):
    """공백/특수문자 제거 + 소문자"""
    if not s or pd.isna(s):
        return ""
    return _RX_NON.sub("", str(s)).lower()

def normalize_period(start, end):
    """기간 정규화: YYYY-MM-DD~YYYY-MM-DD"""
    s = (start or "").strip()
    e = (end or "").strip()
    if not s and not e:
        return ""
    return f"{s}~{e}"

df["norm_title"]    = df["title"].apply(normalize_text)
df["norm_provider"] = df["provider"].apply(normalize_text)
df["norm_period"]   = df.apply(lambda r: normalize_period(r["apply_start"], r["apply_end"]), axis=1)
df["norm_detail_url"] = df["detail_url"].apply(lambda v: str(v).strip().lower())

# dedupe key: 약한 신호 (제목+기관+기간)
df["_dedupe_key"] = df["norm_title"] + "|" + df["norm_provider"] + "|" + df["norm_period"]

print(f"shape: {df.shape}")
empty_str = ""
print(f"norm_title 빈 값: {(df['norm_title']==empty_str).sum()}")
print(f"norm_detail_url 빈 값: {(df['norm_detail_url']==empty_str).sum()}")

shape: (600, 29)
norm_title 빈 값: 0
norm_detail_url 빈 값: 1


## 4. 스코프 룰표 v2 적용

순서:
1. 모든 행 `_scope = main`, `_scope_reason = primary`로 초기화
2. **카테고리 룰** (Priority 1): `s_category=참여/기반` → other
3. **키워드 룰** (Priority 2): 위원회/협의체 등 보강 키워드
4. **dedupe**:
   - 강한 신호 (URL exact) → other (`duplicate_url`)
   - 완전 동일 (제목+기관+기간 일치) → other (`duplicate_exact`)
   - 자매 공고 → review 큐 (scope는 main 유지)

In [5]:
# 1. 초기화
df["_scope"] = "main"
df["_scope_reason"] = "primary"

# 2. 룰표 로드
scope_rule = pd.read_csv(SCOPE_RULE_CSV, comment="#", dtype=str).fillna("")
scope_rule["priority"] = pd.to_numeric(scope_rule["priority"], errors="coerce").fillna(99).astype(int)
scope_rule = scope_rule.sort_values("priority")
print(f"스코프 룰표 shape: {scope_rule.shape}")
display(scope_rule[["rule_id", "scope", "scope_reason", "match_field", "match_type", "pattern", "priority"]])

# 3. 룰 적용
def apply_scope_rule(row):
    """룰을 우선순위 순으로 적용. 첫 매칭에서 결정."""
    for _, rule in scope_rule.iterrows():
        match_field = rule["match_field"]
        match_type = rule["match_type"]
        pattern = rule["pattern"]
        
        if match_field not in row.index:
            continue
        
        target_value = str(row[match_field])
        matched = False
        
        if match_type == "category":
            matched = (target_value == pattern)
        elif match_type == "keyword":
            matched = (pattern in target_value)
        elif match_type == "regex":
            try:
                matched = bool(re.search(pattern, target_value))
            except re.error:
                matched = False
        
        if matched:
            return rule["scope"], rule["scope_reason"]
    
    return "main", "primary"

results = df.apply(apply_scope_rule, axis=1)
df["_scope"] = results.apply(lambda x: x[0])
df["_scope_reason"] = results.apply(lambda x: x[1])

print()
print("=== 룰 적용 후 _scope 분포 ===")
display(df["_scope"].value_counts().rename_axis("scope").reset_index(name="count"))
print()
print("=== _scope_reason 분포 ===")
display(df["_scope_reason"].value_counts().rename_axis("reason").reset_index(name="count"))

스코프 룰표 shape: (6, 8)


,rule_id,scope,scope_reason,match_field,match_type,pattern,priority
0,R01,other,civic_participation,s_category,category,참여/기반,1
1,R10,other,civic_participation,title,keyword,청년참여위원회,2
2,R11,other,civic_participation,title,keyword,청년주권회의,2
3,R12,other,civic_participation,title,keyword,청년원탁회의,2
4,R13,other,civic_participation,title,keyword,청년의회,2
5,R14,other,civic_participation,title,keyword,정책협의체,2



=== 룰 적용 후 _scope 분포 ===


,scope,count
0,main,566
1,other,34



=== _scope_reason 분포 ===


,reason,count
0,primary,566
1,civic_participation,34


## 5. Dedupe 자동 처리

### 강한 신호 (URL exact)
같은 detail_url을 가진 행이 2개 이상이면 → 첫 번째만 main 유지, 나머지 other.

### 완전 동일 (제목+기관+기간)
norm_title + norm_provider + norm_period가 100% 일치하면 → 자동 merge (위와 동일 처리).

### 자매 공고 (애매)
_dedupe_key가 같지만 title이 다른 경우 (예: 파리/도쿄 팝업스토어).
→ scope는 main 유지, `_dup_candidate`에 표시 + review 큐로 분리.

In [6]:
main_only_idx = df[df["_scope"] == "main"].index.tolist()
df["_dup_candidate"] = ""

# === 5.1 강한 신호: URL exact + 제목 동일 ===
# URL만 같고 제목이 다른 경우는 메인페이지 URL 공유 케이스 → 처리 안 함
url_groups = df.loc[main_only_idx][df.loc[main_only_idx, "norm_detail_url"].str.len() > 0].groupby("norm_detail_url")
url_dup_count = 0
url_diff_title_count = 0
for url, g in url_groups:
    if len(g) <= 1:
        continue
    titles = g["norm_title"].unique()
    if len(titles) == 1:
        # URL 같고 제목도 같음 → 진짜 중복 → 자동 처리
        keep_idx = g.index[0]
        drop_idx = g.index[1:]
        df.loc[drop_idx, "_scope"] = "other"
        df.loc[drop_idx, "_scope_reason"] = "duplicate_url"
        df.loc[drop_idx, "_dup_candidate"] = df.loc[keep_idx, "source_id"]
        url_dup_count += len(drop_idx)
    else:
        # URL 같지만 제목 다름 → 메인페이지 URL 공유 케이스 → 처리 안 함
        url_diff_title_count += len(g)
print(f"강한 신호(URL+제목 일치): {url_dup_count}건 자동 처리")
print(f"URL 같지만 제목 다름 (메인페이지 공유): {url_diff_title_count}건 → 처리 안 함")

# === 5.2 완전 동일: norm_title + norm_provider + norm_period 100% 일치 ===
# (URL이 비어있어서 강한 신호로 못 잡힌 케이스 추가 처리)
main_idx_after_url = df[df["_scope"] == "main"].index.tolist()
exact_groups = df.loc[main_idx_after_url].groupby("_dedupe_key")
exact_dup_count = 0
for key, g in exact_groups:
    if len(g) <= 1:
        continue
    if not key or "||" in key:  # 빈 키 또는 부분 빈 키 제외
        continue
    
    # 그룹 내 norm_title이 모두 같으면 → 완전 동일 (자동 merge)
    titles = g["norm_title"].unique()
    if len(titles) == 1 and titles[0]:
        keep_idx = g.index[0]
        drop_idx = g.index[1:]
        df.loc[drop_idx, "_scope"] = "other"
        df.loc[drop_idx, "_scope_reason"] = "duplicate_exact"
        df.loc[drop_idx, "_dup_candidate"] = df.loc[keep_idx, "source_id"]
        exact_dup_count += len(drop_idx)
print(f"완전 동일 자동 처리: {exact_dup_count}건")

# === 5.3 자매 공고: _dedupe_key 같은데 title 다름 → review 큐 ===
main_idx_now = df[df["_scope"] == "main"].index.tolist()
sister_groups = df.loc[main_idx_now].groupby("_dedupe_key")
sister_review_pairs = []
for key, g in sister_groups:
    if len(g) <= 1 or not key or "||" in key:
        continue
    sister_review_pairs.append((key, g.index.tolist()))

sister_count = sum(len(idx_list) for _, idx_list in sister_review_pairs)
print(f"자매 공고 (review 큐): {len(sister_review_pairs)}그룹 / {sister_count}건")

print()
print("=== dedupe 후 최종 _scope 분포 ===")
display(df["_scope"].value_counts().rename_axis("scope").reset_index(name="count"))
print()
print("=== _scope_reason 분포 ===")
display(df["_scope_reason"].value_counts().rename_axis("reason").reset_index(name="count"))

강한 신호(URL+제목 일치): 4건 자동 처리
URL 같지만 제목 다름 (메인페이지 공유): 22건 → 처리 안 함
완전 동일 자동 처리: 8건
자매 공고 (review 큐): 0그룹 / 0건

=== dedupe 후 최종 _scope 분포 ===


,scope,count
0,main,554
1,other,46



=== _scope_reason 분포 ===


,reason,count
0,primary,554
1,civic_participation,34
2,duplicate_exact,8
3,duplicate_url,4


## 6. Review 큐 생성

자매 공고 그룹을 검토자가 엑셀로 열어서 결정할 수 있도록 CSV로 분리.

### 검토자가 채울 컬럼
- `review_decision`: `merge` / `keep_separate` / `unsure`
- `review_canonical`: merge일 경우 어느 source_id를 대표로 할지
- `review_note`: 자유 메모

In [7]:
review_rows = []
for review_idx, (key, idx_list) in enumerate(sister_review_pairs, start=1):
    group = df.loc[idx_list]
    for _, row in group.iterrows():
        review_rows.append({
            "review_id": f"REV{review_idx:04d}",
            "group_size": len(group),
            "source": row["source"],
            "source_id": row["source_id"],
            "title": row["title"],
            "provider": row["provider"],
            "apply_start": row["apply_start"],
            "apply_end": row["apply_end"],
            "norm_title": row["norm_title"],
            "norm_provider": row["norm_provider"],
            "norm_period": row["norm_period"],
            "_dedupe_key": row["_dedupe_key"],
            # 검토자가 채울 컬럼
            "review_decision": "",  # merge / keep_separate / unsure
            "review_canonical": "",
            "review_note": "",
        })

review_df = pd.DataFrame(review_rows)

if len(review_df):
    review_df.to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")
    print(f"review 큐 저장: {REVIEW_CSV}")
    print(f"총 {len(review_df)}건 ({len(sister_review_pairs)}개 그룹)")
    
    print()
    print("=== review 큐 미리보기 (전체 그룹) ===")
    for rid in review_df["review_id"].unique():
        g = review_df[review_df["review_id"] == rid]
        print(f"\n[{rid}] group_size={g['group_size'].iloc[0]}")
        display(g[["source", "source_id", "title", "provider"]])
else:
    print("자매 공고 그룹 없음. review 큐 생성 안 함.")

자매 공고 그룹 없음. review 큐 생성 안 함.


## 7. 임베딩용 main_csv 저장

`_scope=main`인 행만 추려서 임베딩 단계로 넘길 수 있는 깨끗한 CSV 생성.

### 컬럼 구성
- 식별: `source`, `source_id`
- 표시용: `title`, `summary`, `s_category`, `provider`, `region`
- raw 메타: `target_group`, `target_age`, `support_type`, `apply_start`, `apply_end`, `detail_url`
- 스코프: `_scope`, `_scope_reason`
- 정규화: `norm_title`, `norm_provider`, `norm_period`

In [8]:
main_df = df[df["_scope"] == "main"].copy()

MAIN_COLS = [
    # 식별
    "source", "source_id",
    # 서비스 표시용 (임베딩 대상)
    "title", "summary", "s_category", "provider", "region",
    # raw 메타 (셀프쿼리용 메타데이터)
    "target_group", "target_age", "target_detail",
    "income_condition", "startup_stage", "support_type",
    "apply_start", "apply_end",
    "additional_conditions", "required_documents", "application_method",
    "detail_url",
    # 스코프 정보
    "_scope", "_scope_reason",
    # 정규화 (dedupe 추적용)
    "norm_title", "norm_provider", "norm_period",
    # 원본 카테고리 (룰표 변경 시 재적용 가능하도록 보존)
    "category", "subcategory",
]

# 누락 컬럼은 빈 문자열로
for col in MAIN_COLS:
    if col not in main_df.columns:
        main_df[col] = ""

main_out = main_df[MAIN_COLS].copy()
main_out.to_csv(MAIN_CSV, index=False, encoding="utf-8-sig")

print(f"main 저장: {MAIN_CSV}")
print(f"shape: {main_out.shape}")
print()
print("=== 최종 s_category × source (main만) ===")
display(pd.crosstab(main_out["s_category"], main_out["source"], margins=True, margins_name="합계"))
print()
print("=== 미리보기 ===")
display(main_out.head(3))

main 저장: c:\Users\CY2\github\policyrec\PolicyRec\data\csv\main\main_v1_1_5.csv
shape: (554, 26)

=== 최종 s_category × source (main만) ===


source,biz,kst,youth,합계
s_category,,,,
경영,64,0,0,64
교육/멘토링,0,44,18,62
기술,39,4,0,43
기타,2,0,0,2
복지/문화,0,0,55,55
시설/공간,0,30,0,30
인력/일자리,18,2,62,82
자금,8,1,0,9
주거,0,0,22,22



=== 미리보기 ===


,source,source_id,title,summary,s_category,provider,region,target_group,target_age,target_detail,...,required_documents,application_method,detail_url,_scope,_scope_reason,norm_title,norm_provider,norm_period,category,subcategory
0,biz,PBLN_000000000121348,2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고,"<p>2026년 한-체코,한-중국 에너지국제공동연구사업의 신규지원 대상 연구개발과제...",기술,한국에너지기술평가원,기후에너지환경부,중소기업,,,...,온라인 접수(범부처통합연구지원시스템),온라인 접수(범부처통합연구지원시스템),https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,2026년한체코ㆍ한중국에너지국제공동rd신규지원대상과제공고,한국에너지기술평가원,사업별 상이~,기술,공동기술개발
1,biz,PBLN_000000000121347,[경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고,<p>한국생산기술연구원 첨단하이브리드생산기술센터에서는 동부경남지역 소재부품분야 관련...,기술,한국생산기술연구원,경상남도,중소기업,,,...,이메일 접수 (hsb85@kitech.re.kr),이메일 접수 (hsb85@kitech.re.kr),https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,경남2026년소재부품성장잠재기업육성사업수요기업모집공고,한국생산기술연구원,2026-04-23~2026-05-07,기술,기술사업화/이전/지도
2,biz,PBLN_000000000121346,[경기] 이천시 2026년 도ㆍ공예기업 맞춤형 온라인 마케팅 지원사업 참여기업 모집 공고,<p>이천시와 경기테크노파크에서는 관내 도ㆍ공예기업 우수제품의 온라인 매출 증대와 ...,판로/수출,경기테크노파크,경기도,중소기업,,,...,"온라인, 이메일, 방문, 우편 접수\r\n- 접수처 : (15588) 경기도 안산시...","온라인, 이메일, 방문, 우편 접수\r\n- 접수처 : (15588) 경기도 안산시...",https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,경기이천시2026년도ㆍ공예기업맞춤형온라인마케팅지원사업참여기업모집공고,경기테크노파크,2026-04-23~2026-05-22,내수,홍보지원


## 8. 요약 + 다음 단계

### 처리 결과 요약

In [9]:
total = len(df)
main_count = (df["_scope"] == "main").sum()
other_count = (df["_scope"] == "other").sum()

reason_counts = df["_scope_reason"].value_counts()

print("=" * 60)
print("PolicyRec v1.1.5 처리 결과")
print("=" * 60)
print(f"총 raw 데이터: {total}건")
print(f"  → main (임베딩 대상): {main_count}건")
print(f"  → other: {other_count}건")
print()
print("[other 상세]")
for reason, cnt in reason_counts.items():
    if reason != "primary":
        print(f"  - {reason}: {cnt}건")
print()
print(f"review 큐: {len(sister_review_pairs)}그룹 / {sister_count}건")
print()
print("=" * 60)
print("다음 단계")
print("=" * 60)
print(f"1. 임베딩 노트북에서 {MAIN_CSV.name} 입력으로 사용")
print(f"   - 임베딩 대상 텍스트: title + summary + s_category + region")
print(f"   - 셀프쿼리 메타데이터: s_category, region, target_age, target_group 등")
print(f"2. (선택) review 큐 검토 후 추가 dedupe 반영")
print(f"   - {REVIEW_CSV.name}에 review_decision 채우기")
print(f"   - 별도 후처리 노트북에서 반영")

PolicyRec v1.1.5 처리 결과
총 raw 데이터: 600건
  → main (임베딩 대상): 554건
  → other: 46건

[other 상세]
  - civic_participation: 34건
  - duplicate_exact: 8건
  - duplicate_url: 4건

review 큐: 0그룹 / 0건

다음 단계
1. 임베딩 노트북에서 main_v1_1_5.csv 입력으로 사용
   - 임베딩 대상 텍스트: title + summary + s_category + region
   - 셀프쿼리 메타데이터: s_category, region, target_age, target_group 등
2. (선택) review 큐 검토 후 추가 dedupe 반영
   - review_queue_v1_1_5.csv에 review_decision 채우기
   - 별도 후처리 노트북에서 반영


## 9. 임베딩 단계 인수인계

### 임베딩에 사용할 텍스트 컬럼
다음 컬럼을 조합해서 임베딩 텍스트 생성 권장:

```python
def build_embedding_text(row):
    parts = [
        f"제목: {row['title']}",
        f"카테고리: {row['s_category']}",
        f"지역: {row['region']}",
        f"대상: {row['target_group']}",
        f"요약: {row['summary']}",
    ]
    return "\n".join(parts)
```

### 셀프쿼리용 메타데이터 컬럼
Supabase 적재 시 별도 컬럼으로 보관 (필터 쿼리에 사용):
- `s_category`: 카테고리 필터
- `region`: 지역 필터
- `target_age`: 연령 필터
- `target_group`: 대상자 필터
- `apply_start`, `apply_end`: 모집 기간 필터
- `_scope_reason`: 디버깅/통계용

### 룰표 수정 흐름
1. `cat_rule_v1_1_5.csv` 또는 `scope_rule_v1_1_5.csv` 수정
2. 이 노트북 재실행
3. main_csv 새로 생성 → 임베딩 노트북도 재실행